In [1]:
import heapq
from collections import deque
import time

class Node:
    def __init__(self, position, heuristic):
        self.position = position
        self.heuristic = heuristic

    def __lt__(self, other):
        return self.heuristic < other.heuristic


def reconstruct_path(path_dict, start, goal):
    current = goal
    path = []
    while current is not None:
        path.append(current)
        current = path_dict[current]
    return path[::-1]


class TreasureHunt:
    def __init__(self, grid_size, treasure_pos, start_pos):
        self.grid_size = grid_size
        self.treasure_pos = treasure_pos
        self.start_pos = start_pos

    def manhattan_distance(self, pos):
        return abs(pos[0] - self.treasure_pos[0]) + abs(pos[1] - self.treasure_pos[1])

    def get_neighbors(self, pos):
        x, y = pos
        neighbors = []
        for dx, dy in [(0,1), (1,0), (0,-1), (-1,0)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                neighbors.append((nx, ny))
        return neighbors

    def greedy_best_first_search(self):
        pq = []
        heapq.heappush(pq, Node(self.start_pos, self.manhattan_distance(self.start_pos)))
        visited = set()
        parent = {self.start_pos: None}
        explored = 0

        while pq:
            node = heapq.heappop(pq)
            pos = node.position

            if pos == self.treasure_pos:
                return reconstruct_path(parent, self.start_pos, self.treasure_pos), explored

            if pos in visited:
                continue

            visited.add(pos)
            explored += 1

            for n in self.get_neighbors(pos):
                if n not in visited:
                    heapq.heappush(pq, Node(n, self.manhattan_distance(n)))
                    if n not in parent:
                        parent[n] = pos

        return None, explored

    def bfs(self):
        q = deque([(self.start_pos, [self.start_pos])])
        visited = {self.start_pos}
        explored = 0

        while q:
            pos, path = q.popleft()
            explored += 1

            if pos == self.treasure_pos:
                return path, explored

            for n in self.get_neighbors(pos):
                if n not in visited:
                    visited.add(n)
                    q.append((n, path + [n]))

        return None, explored

    def dfs(self):
        stack = [(self.start_pos, [self.start_pos])]
        visited = set()
        explored = 0

        while stack:
            pos, path = stack.pop()

            if pos in visited:
                continue

            visited.add(pos)
            explored += 1

            if pos == self.treasure_pos:
                return path, explored

            for n in self.get_neighbors(pos):
                if n not in visited:
                    stack.append((n, path + [n]))

        return None, explored


def main():
    n = int(input("Grid size: "))
    sx = int(input("Start X: "))
    sy = int(input("Start Y: "))
    tx = int(input("Treasure X: "))
    ty = int(input("Treasure Y: "))

    game = TreasureHunt(n, (tx, ty), (sx, sy))

    t0 = time.time()
    g_path, g_nodes = game.greedy_best_first_search()
    g_time = time.time() - t0

    t0 = time.time()
    b_path, b_nodes = game.bfs()
    b_time = time.time() - t0

    t0 = time.time()
    d_path, d_nodes = game.dfs()
    d_time = time.time() - t0

    print("\nGBFS", len(g_path) if g_path else -1, g_nodes, f"{g_time:.6f}")
    print("BFS ", len(b_path) if b_path else -1, b_nodes, f"{b_time:.6f}")
    print("DFS ", len(d_path) if d_path else -1, d_nodes, f"{d_time:.6f}")

if __name__ == "__main__":
    main()


Grid size: 10
Start X: 4
Start Y: 6
Treasure X: 9
Treasure Y: 2

GBFS 10 9 0.000104
BFS  10 92 0.000211
DFS  38 38 0.000076


In [3]:
import heapq
import time

GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)


def manhattan(state):
    dist = 0
    for i in range(9):
        if state[i] != 0:
            goal_pos = GOAL.index(state[i])
            dist += abs(i // 3 - goal_pos // 3) + abs(i % 3 - goal_pos % 3)
    return dist


def misplaced(state):
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != GOAL[i])


def get_neighbors(state):
    neighbors = []
    zero = state.index(0)
    x, y = zero // 3, zero % 3

    moves = [(-1,0),(1,0),(0,-1),(0,1)]
    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_zero = nx * 3 + ny
            new_state = list(state)
            new_state[zero], new_state[new_zero] = new_state[new_zero], new_state[zero]
            neighbors.append(tuple(new_state))

    return neighbors


def is_solvable(state):
    inv = 0
    arr = [x for x in state if x != 0]
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0


def gbfs(start, heuristic):
    pq = []
    heapq.heappush(pq, (heuristic(start), start))
    parent = {start: None}
    visited = set()
    explored = 0

    while pq:
        _, current = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)
        explored += 1

        if current == GOAL:
            path = []
            while current:
                path.append(current)
                current = parent[current]
            return path[::-1], explored

        for n in get_neighbors(current):
            if n not in visited:
                if n not in parent:
                    parent[n] = current
                heapq.heappush(pq, (heuristic(n), n))

    return None, explored


def main():
    start = tuple(map(int, input("Enter puzzle (0 for blank): ").split()))

    if not is_solvable(start):
        print("Unsolvable")
        return

    t0 = time.time()
    path_m, nodes_m = gbfs(start, manhattan)
    t1 = time.time()

    t2 = time.time()
    path_h, nodes_h = gbfs(start, misplaced)
    t3 = time.time()

    print("Manhattan", len(path_m) - 1, nodes_m, f"{t1 - t0:.6f}")
    print("Misplaced", len(path_h) - 1, nodes_h, f"{t3 - t2:.6f}")


if __name__ == "__main__":
    main()


Enter puzzle (0 for blank): 1 2 3 4 5 6 7 8 0
Manhattan 0 1 0.000026
Misplaced 0 1 0.000013
